# Final experiment: regularized logistic regression

## 1. Introduction

This final experiment tests whether a small, predeclared set of L2-regularization strengths improves the current M1 logistic-regression model. It keeps M1’s saved feature list and the original real-data train/test split.

1. Read the actual feature list from the saved M1 bundle and verify it against M2.
2. Compare the unregularized baseline (`C=np.inf`) with five L2 strengths using nested, stratified cross-validation on training rows only.
3. Choose a regularization strength and a threshold for the 85% recall goal using training-only out-of-fold predictions.
4. Fit the selected candidate on the full training set, then make one final comparison with M1 on the fixed test set.

**Source files:** `diabetes_model_train.csv`, `diabetes_model_test.csv`, and the saved M1/M2 bundles. The test labels are not used for feature selection, hyperparameter choice, or threshold choice.

**Success rule:** recall of at least 85% is the requirement; among models meeting it, compare specificity first, then report balanced-test precision and approximate precision at 14% prevalence. The fixed test set is used once, after choices are fixed.

## 2. Why regularization

Regularization adds a penalty for large logistic-regression coefficients. It can reduce overfitting when predictors overlap or carry noisy signals. The strength is controlled by `C`: smaller `C` means stronger L2 regularization; larger `C` means weaker regularization. `C=np.inf` is the project’s unregularized baseline.

The tested values are fixed in advance: `0.01`, `0.1`, `1`, `10`, and `100`. Scaling is fitted inside each training fold. For each outer fold, inner out-of-fold predictions set the threshold and choose a regularized candidate; the outer fold estimates the result of that selection procedure. No test rows enter this process.

## 3. Imports and metrics

Use the project’s fixed seed, the saved M1/M2 features, and the same evaluation definitions used in the other experiments.

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TARGET = "Diabetes_binary"
RANDOM_STATE = 42
TARGET_RECALL = 0.85
REAL_PREVALENCE = 0.14
REGULARIZATION_C = [0.01, 0.1, 1.0, 10.0, 100.0]
OUTER_FOLDS = 5
INNER_FOLDS = 5

ROOT = Path("../..")
DATA = ROOT / "data" / "05_cmb_train_test_for_models"
MODELS = ROOT / "application" / "models"
RESULTS = Path("results")
RESULTS.mkdir(parents=True, exist_ok=True)
NON_BINARY = {"BMI", "GenHlth", "MentHlth", "PhysHlth", "Age", "Income"}

def make_logreg(features, C):
    """Fit scaling inside the pipeline and use the requested L2 penalty strength."""
    scaled = [name for name in features if name in NON_BINARY]
    prep = ColumnTransformer(
        [("scale", StandardScaler(), scaled)],
        remainder="passthrough",
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(C=C, max_iter=2000)),
    ])

def threshold_for(y_true, probabilities, goal=TARGET_RECALL):
    """Highest threshold reaching the recall goal on training OOF predictions."""
    _, recall, thresholds = precision_recall_curve(y_true, probabilities)
    eligible = thresholds[recall[:-1] >= goal]
    if len(eligible) == 0:
        raise ValueError(f"No threshold reaches the requested recall goal ({goal:.0%}).")
    return float(eligible.max())

def evaluate(y_true, probabilities, threshold):
    """Return the shared project metrics at a fixed threshold."""
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    return {
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_true, predicted),
        "Recall": tp / (tp + fn) if tp + fn else np.nan,
        "Specificity": tn / (tn + fp) if tn + fp else np.nan,
        "Precision (balanced test)": tp / (tp + fp) if tp + fp else np.nan,
        "F1": f1_score(y_true, predicted, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, probabilities),
        "Brier score": brier_score_loss(y_true, probabilities),
        "False negatives": fn,
        "False positives": fp,
        "Share flagged": float(predicted.mean()),
    }

def adjusted_precision(recall, specificity, prevalence=REAL_PREVALENCE):
    """Approximate precision and share flagged at the assumed population prevalence."""
    share_flagged = recall * prevalence + (1 - specificity) * (1 - prevalence)
    precision = recall * prevalence / share_flagged if share_flagged else np.nan
    return precision, share_flagged

print(f"scikit-learn: {sklearn.__version__}")

scikit-learn: 1.9.1


## 4. Load saved models and fixed real partitions

Read M1/M2’s actual feature lists instead of relying on older notes. The experiment stops if their features or order do not match.

In [2]:
m1_bundle = joblib.load(MODELS / "logreg_70k.joblib")
m2_bundle = joblib.load(MODELS / "logreg_200k.joblib")
FEATURES = list(m1_bundle["features"])
FEATURES_M2 = list(m2_bundle["features"])
if FEATURES != FEATURES_M2:
    raise ValueError("M1 and M2 feature names/order differ; stop and resolve before comparison.")

train = pd.read_csv(DATA / "diabetes_model_train.csv")
test = pd.read_csv(DATA / "diabetes_model_test.csv")
for name, frame in [("training", train), ("test", test)]:
    missing = sorted(set(FEATURES + [TARGET]) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} file is missing columns: {missing}")
    if frame[FEATURES + [TARGET]].isna().any().any():
        raise ValueError(f"Missing values found in {name} data.")

feature_report = pd.DataFrame([
    {"Model": "M1 saved bundle", "Feature count": len(FEATURES),
     "Features": ", ".join(FEATURES), "Saved threshold": m1_bundle["threshold"]},
    {"Model": "M2 saved bundle", "Feature count": len(FEATURES_M2),
     "Features": ", ".join(FEATURES_M2), "Saved threshold": m2_bundle["threshold"]},
])
display(feature_report.style.hide(axis="index"))
print(f"Training rows: {len(train):,}; fixed test rows: {len(test):,}.")
print("The test table is loaded but its labels are not used until the final comparison.")

Model,Feature count,Features,Saved threshold
M1 saved bundle,14,"HighBP, HighChol, BMI, Stroke, HeartDiseaseorAttack, Fruits, HvyAlcoholConsump, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Income",0.396314
M2 saved bundle,14,"HighBP, HighChol, BMI, Stroke, HeartDiseaseorAttack, Fruits, HvyAlcoholConsump, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Income",0.375532


Training rows: 56,155; fixed test rows: 14,039.
The test table is loaded but its labels are not used until the final comparison.


## 5. Nested training-only CV: select a C and threshold inside each outer fold

For every outer fold, the inner folds generate out-of-fold scores for the unregularized baseline and each regularized model. Each candidate gets its own training-only threshold for the recall goal. The best regularized candidate is chosen by inner-fold specificity; it is selected only if its specificity exceeds the unregularized baseline. Otherwise, that outer fold retains the unregularized baseline. The outer validation fold then scores this selection procedure.

This nested layout keeps the outer validation fold out of threshold and hyperparameter selection. Its reported recall can fall below 85% because the 85% target is estimated from training folds, not forced on validation data.

In [3]:
def candidate_specs():
    """Names and strengths tested in this experiment, including unregularized reference."""
    specs = {"Unregularized reference": np.inf}
    specs.update({f"L2 C={c:g}": c for c in REGULARIZATION_C})
    return specs

def oof_probabilities(frame, features, specs, n_splits, seed):
    """Build out-of-fold training probabilities for every predeclared C value."""
    y = frame[TARGET].to_numpy()
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = {name: np.full(len(frame), np.nan) for name in specs}
    for fold, (fit_idx, valid_idx) in enumerate(splitter.split(frame[features], y), start=1):
        fit_frame = frame.iloc[fit_idx]
        valid_x = frame.iloc[valid_idx][features]
        for name, C in specs.items():
            model = make_logreg(features, C)
            model.fit(fit_frame[features], fit_frame[TARGET])
            out[name][valid_idx] = model.predict_proba(valid_x)[:, 1]
    if any(np.isnan(prob).any() for prob in out.values()):
        raise AssertionError("Some OOF predictions were not created.")
    return out

def choose_regularized_candidate(y, oof, specs):
    """Choose best regularized C by OOF specificity, using the unregularized model as a gate."""
    summaries = {}
    for name, probabilities in oof.items():
        cutoff = threshold_for(y, probabilities)
        metrics = evaluate(y, probabilities, cutoff)
        summaries[name] = {**metrics, "C": specs[name]}
    baseline_specificity = summaries["Unregularized reference"]["Specificity"]
    regularized = [name for name in specs if name != "Unregularized reference"]
    best_name = max(regularized, key=lambda name: (
        summaries[name]["Specificity"], specs[name]
    ))
    # If regularization does not improve training-only specificity, retain the unregularized baseline.
    selected = best_name if summaries[best_name]["Specificity"] > baseline_specificity + 1e-12 else "Unregularized reference"
    return selected, summaries

specs = candidate_specs()
outer_splitter = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
outer_rows = []
selected_c_counts = {}

for outer_fold, (outer_train_idx, outer_valid_idx) in enumerate(outer_splitter.split(train[FEATURES], train[TARGET]), start=1):
    outer_train = train.iloc[outer_train_idx].reset_index(drop=True)
    outer_valid = train.iloc[outer_valid_idx].reset_index(drop=True)
    inner_oof = oof_probabilities(
        outer_train, FEATURES, specs, INNER_FOLDS, RANDOM_STATE + outer_fold
    )
    y_inner = outer_train[TARGET].to_numpy()
    selected_name, inner_summaries = choose_regularized_candidate(y_inner, inner_oof, specs)
    selected_c_counts[selected_name] = selected_c_counts.get(selected_name, 0) + 1

    selected_threshold = threshold_for(y_inner, inner_oof[selected_name])
    selected_C = specs[selected_name]
    selected_model = make_logreg(FEATURES, selected_C)
    selected_model.fit(outer_train[FEATURES], outer_train[TARGET])
    selected_probability = selected_model.predict_proba(outer_valid[FEATURES])[:, 1]
    selected_metrics = evaluate(outer_valid[TARGET].to_numpy(), selected_probability, selected_threshold)
    outer_rows.append({"Outer fold": outer_fold, "Evaluation": "Nested selection policy",
                       "Selected candidate": selected_name, "C": selected_C,
                       "Training rows": len(outer_train), **selected_metrics})

    # Score the fold-matched unregularized reference using its own inner-CV threshold.
    baseline_name = "Unregularized reference"
    baseline_threshold = threshold_for(y_inner, inner_oof[baseline_name])
    baseline_model = make_logreg(FEATURES, np.inf)
    baseline_model.fit(outer_train[FEATURES], outer_train[TARGET])
    baseline_probability = baseline_model.predict_proba(outer_valid[FEATURES])[:, 1]
    baseline_metrics = evaluate(outer_valid[TARGET].to_numpy(), baseline_probability, baseline_threshold)
    outer_rows.append({"Outer fold": outer_fold, "Evaluation": "Unregularized reference",
                       "Selected candidate": baseline_name, "C": np.inf,
                       "Training rows": len(outer_train), **baseline_metrics})
    print(f"Completed outer fold {outer_fold}/{OUTER_FOLDS}; selection chose {selected_name}.")

outer_results = pd.DataFrame(outer_rows)
outer_summary = (outer_results.groupby("Evaluation", as_index=False)
                 .agg(**{metric: (metric, "mean") for metric in
                         ["Recall", "Specificity", "Precision (balanced test)", "ROC AUC", "Threshold"]},
                      **{"Recall SD": ("Recall", "std"), "Specificity SD": ("Specificity", "std")}))
display(outer_results.style.format(precision=3).hide(axis="index"))
display(outer_summary.style.format(precision=3).hide(axis="index"))
print("Inner-fold selections:", selected_c_counts)
print("These are outer-fold estimates from training rows; the fixed test labels remain unused.")

Completed outer fold 1/5; selection chose L2 C=10.


Completed outer fold 2/5; selection chose L2 C=0.01.


Completed outer fold 3/5; selection chose L2 C=0.1.


Completed outer fold 4/5; selection chose L2 C=0.01.


Completed outer fold 5/5; selection chose L2 C=0.1.


Outer fold,Evaluation,Selected candidate,C,Training rows,Threshold,Accuracy,Recall,Specificity,Precision (balanced test),F1,ROC AUC,Brier score,False negatives,False positives,Share flagged
1,Nested selection policy,L2 C=10,10.000,44924,0.397,0.729,0.846,0.613,0.686,0.758,0.808,0.178,863,2175,0.617
1,Unregularized reference,Unregularized reference,inf,44924,0.397,0.729,0.846,0.613,0.686,0.758,0.808,0.178,863,2175,0.617
2,Nested selection policy,L2 C=0.01,0.010,44924,0.399,0.730,0.852,0.608,0.685,0.759,0.804,0.180,831,2203,0.622
2,Unregularized reference,Unregularized reference,inf,44924,0.395,0.730,0.853,0.607,0.684,0.760,0.804,0.180,825,2208,0.623
3,Nested selection policy,L2 C=0.1,0.100,44924,0.397,0.732,0.853,0.612,0.687,0.761,0.812,0.176,827,2180,0.621
3,Unregularized reference,Unregularized reference,inf,44924,0.396,0.732,0.853,0.611,0.687,0.761,0.812,0.176,824,2183,0.621
4,Nested selection policy,L2 C=0.01,0.010,44924,0.399,0.728,0.849,0.606,0.683,0.757,0.806,0.179,848,2210,0.621
4,Unregularized reference,Unregularized reference,inf,44924,0.396,0.729,0.851,0.606,0.684,0.758,0.806,0.179,836,2211,0.622
5,Nested selection policy,L2 C=0.1,0.100,44924,0.398,0.722,0.845,0.600,0.679,0.753,0.806,0.179,870,2247,0.623
5,Unregularized reference,Unregularized reference,inf,44924,0.398,0.722,0.845,0.599,0.678,0.753,0.806,0.179,869,2251,0.623


Evaluation,Recall,Specificity,Precision (balanced test),ROC AUC,Threshold,Recall SD,Specificity SD
Nested selection policy,0.849,0.608,0.684,0.807,0.398,0.003,0.005
Unregularized reference,0.850,0.607,0.684,0.807,0.397,0.004,0.005


Inner-fold selections: {'L2 C=10': 1, 'L2 C=0.01': 2, 'L2 C=0.1': 2}
These are outer-fold estimates from training rows; the fixed test labels remain unused.


## 6. Select the final regularization strength and threshold from training-only OOF scores

After estimating the selection procedure with nested CV, build new five-fold OOF predictions on all training rows. Select the regularized strength with the highest OOF specificity at its training-only 85%-recall threshold. The test set is not used to choose the candidate or threshold. The unregularized OOF result is retained as the reference.

In [4]:
full_oof = oof_probabilities(train, FEATURES, specs, OUTER_FOLDS, RANDOM_STATE + 100)
full_oof_selected_name, full_oof_summaries = choose_regularized_candidate(
    train[TARGET].to_numpy(), full_oof, specs
)

full_cv_rows = []
for name, summary in full_oof_summaries.items():
    full_cv_rows.append({"Candidate": name, **summary})
full_cv_table = pd.DataFrame(full_cv_rows)
# Put the unregularized reference first, then regularized models ordered by C.
full_cv_table["_order"] = full_cv_table["Candidate"].map(
    {name: i for i, name in enumerate(specs)}
)
full_cv_table = full_cv_table.sort_values("_order").drop(columns="_order")
display(full_cv_table.style.format(precision=3).hide(axis="index"))

FINAL_NAME = max(
    [name for name in specs if name != "Unregularized reference"],
    key=lambda name: (full_oof_summaries[name]["Specificity"], specs[name]),
)
FINAL_C = specs[FINAL_NAME]
FINAL_THRESHOLD = threshold_for(train[TARGET].to_numpy(), full_oof[FINAL_NAME])
print(f"Training-only selected regularized candidate: {FINAL_NAME}; threshold={FINAL_THRESHOLD:.4f}.")
print(f"Training-only selection policy retained: {full_oof_selected_name}.")

# Save training-only diagnostics before the final comparison.
outer_results.to_csv(RESULTS / "03_regularized_logreg_nested_cv_folds.csv", index=False)
outer_summary.to_csv(RESULTS / "03_regularized_logreg_nested_cv_summary.csv", index=False)
full_cv_table.to_csv(RESULTS / "03_regularized_logreg_full_training_oof_grid.csv", index=False)

Candidate,Threshold,Accuracy,Recall,Specificity,Precision (balanced test),F1,ROC AUC,Brier score,False negatives,False positives,Share flagged,C
Unregularized reference,0.396,0.728,0.850,0.607,0.684,0.758,0.807,0.178,4211,11043,0.622,inf
L2 C=0.01,0.399,0.728,0.850,0.606,0.684,0.758,0.807,0.179,4211,11049,0.622,0.010
L2 C=0.1,0.397,0.728,0.850,0.607,0.684,0.758,0.807,0.178,4211,11036,0.622,0.100
L2 C=1,0.396,0.728,0.850,0.607,0.684,0.758,0.807,0.178,4211,11042,0.622,1.000
L2 C=10,0.396,0.728,0.850,0.607,0.684,0.758,0.807,0.178,4211,11043,0.622,10.000
L2 C=100,0.396,0.728,0.850,0.607,0.684,0.758,0.807,0.178,4211,11043,0.622,100.000


Training-only selected regularized candidate: L2 C=0.1; threshold=0.3967.
Training-only selection policy retained: L2 C=0.1.


## 7. Fit the selected candidate on all training rows

Fit the chosen L2 model on the fixed training partition. Keep this model inside the notebook only; do not overwrite the app’s saved M1 bundle.

In [5]:
final_candidate = make_logreg(FEATURES, FINAL_C)
final_candidate.fit(train[FEATURES], train[TARGET])
print(f"Final experiment model fit with {len(train):,} real training rows and C={FINAL_C:g}.")
print("No model file was written to application/models/.")

Final experiment model fit with 56,155 real training rows and C=0.1.
No model file was written to application/models/.


## 8. One final comparison on the fixed test set

The model and threshold are fixed. Now score M1 at its saved threshold and the selected regularized model at its training-only threshold. Do not adjust the candidate after seeing these test results.

In [6]:
X_test = test[FEATURES]
y_test = test[TARGET].to_numpy()
m1_probability = m1_bundle["model"].predict_proba(X_test)[:, 1]
candidate_probability = final_candidate.predict_proba(X_test)[:, 1]

final_rows = []
for name, probabilities, cutoff in [
    ("M1 saved unregularized model", m1_probability, float(m1_bundle["threshold"])),
    (f"Selected L2 model ({FINAL_NAME})", candidate_probability, FINAL_THRESHOLD),
]:
    row = {"Model": name, **evaluate(y_test, probabilities, cutoff)}
    row["Selected C"] = np.inf if name.startswith("M1") else FINAL_C
    row["Approx. precision at 14% prevalence"] = adjusted_precision(row["Recall"], row["Specificity"])[0]
    row["Approx. share flagged at 14% prevalence"] = adjusted_precision(row["Recall"], row["Specificity"])[1]
    final_rows.append(row)
final_results = pd.DataFrame(final_rows)
columns = ["Model", "Selected C", "Threshold", "Accuracy", "Recall", "Specificity",
           "Precision (balanced test)", "Approx. precision at 14% prevalence",
           "Share flagged", "Approx. share flagged at 14% prevalence", "ROC AUC", "Brier score",
           "False negatives", "False positives"]
display(final_results[columns].style.format(precision=3).hide(axis="index"))
final_results[columns].to_csv(RESULTS / "03_regularized_logreg_final_test_comparison.csv", index=False)

Model,Selected C,Threshold,Accuracy,Recall,Specificity,Precision (balanced test),Approx. precision at 14% prevalence,Share flagged,Approx. share flagged at 14% prevalence,ROC AUC,Brier score,False negatives,False positives
M1 saved unregularized model,inf,0.396,0.726,0.850,0.602,0.681,0.258,0.624,0.461,0.800,0.182,1052,2793
Selected L2 model (L2 C=0.1),0.100,0.397,0.726,0.850,0.602,0.681,0.258,0.624,0.461,0.800,0.182,1053,2793


## 9. Paired uncertainty: selected regularized candidate versus M1

Bootstrap the same test respondents for both models. The thresholds remain fixed at their training-selected values. Interpret these intervals as uncertainty for this fixed test sample, not evidence of clinical validity.

In [7]:
rng = np.random.default_rng(RANDOM_STATE)
BOOTSTRAPS = 1_000
m1_pred = (m1_probability >= float(m1_bundle["threshold"])).astype(int)
candidate_pred = (candidate_probability >= FINAL_THRESHOLD).astype(int)
boot_rows = []
for _ in range(BOOTSTRAPS):
    idx = rng.integers(0, len(y_test), size=len(y_test))
    y_b = y_test[idx]
    p_m1, p_candidate = m1_pred[idx], candidate_pred[idx]
    pos, neg = y_b == 1, y_b == 0
    if not pos.any() or not neg.any():
        continue
    boot_rows.append({
        "Recall difference (candidate - M1)": (p_candidate[pos] == 1).mean() - (p_m1[pos] == 1).mean(),
        "Specificity difference (candidate - M1)": (p_candidate[neg] == 0).mean() - (p_m1[neg] == 0).mean(),
    })
boot = pd.DataFrame(boot_rows)
uncertainty_rows = []
for column in boot.columns:
    point = (final_results.loc[1, "Recall"] - final_results.loc[0, "Recall"]
             if column.startswith("Recall") else
             final_results.loc[1, "Specificity"] - final_results.loc[0, "Specificity"])
    uncertainty_rows.append({"Difference": column, "Observed difference": point,
                             "95% lower": boot[column].quantile(0.025),
                             "95% upper": boot[column].quantile(0.975)})
uncertainty = pd.DataFrame(uncertainty_rows)
display(uncertainty.style.format(precision=3).hide(axis="index"))
uncertainty.to_csv(RESULTS / "03_regularized_logreg_paired_bootstrap.csv", index=False)

Difference,Observed difference,95% lower,95% upper
Recall difference (candidate - M1),-0.000,-0.001,0.000
Specificity difference (candidate - M1),0.000,-0.001,0.001


## 10. Conclusion and limitations

**Observed run (seed 42):** training-only OOF selection chose L2 `C=0.1` as the strongest regularized candidate by specificity. In the final comparison, M1 reached **85.01% recall**, **60.21% specificity**, and **68.12% balanced-test precision**. The selected regularized candidate reached **85.00% recall**, **60.21% specificity**, and **68.11% balanced-test precision**. Its ROC AUC was effectively unchanged (**0.80032 vs 0.80031**).

The candidate missed the 85% recall requirement by a very small amount. Its specificity difference from M1 was **0.00 percentage points** (paired bootstrap 95% interval: **−0.057 to +0.057 points**), so there is no evidence of improved specificity. **Keep the saved M1 model.** Regularization did not produce a practical improvement in this experiment.

**Decision:** keep M1 unless a candidate reaches at least 85% recall on the final test and improves specificity over M1 with a paired 95% interval entirely above zero. Do not adjust this candidate after viewing the test results.

**Limitations:** this is one fixed 2015 U.S. survey split; answers are self-reported; the outcome is previously diagnosed diabetes; balanced-test precision is optimistic for the population; the 14% adjustment is approximate; and model performance is not clinical validation. The final test set represents this dataset, not a new population.

**Next step:** document M1 as the selected app model and stop tuning unless new real data or a justified clinical requirement becomes available.